# IOAI — 2026 Contest Ticket Extraction (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-ticket-extraction/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Ticket Extraction — 모범답안 (지도학습 + 제품명 매칭)

고정 어휘 필드(category/priority/sentiment)는 **문자 n-gram TF-IDF + LinearSVC** 로 분류하고, 자유형
`product` 는 학습셋에 등장한 제품명 목록에서 **텍스트에 나타난 가장 긴 이름**을 매칭한다. 전체 정확도 ≈ 0.88
(category 1.0 / priority·sentiment 0.9 / product 0.7).

In [ ]:
import pandas as pd
train = pd.read_csv("data/train.csv")     # id, text, category, priority, product, sentiment
test  = pd.read_csv("data/test.csv")      # id, text
FIELDS = ["category", "priority", "product", "sentiment"]
print("train", len(train), "| test", len(test))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
preds = {}
for f in ["category", "priority", "sentiment"]:
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3), min_df=2).fit(train.text)
    clf = LinearSVC().fit(vec.transform(train.text), train[f])
    preds[f] = clf.predict(vec.transform(test.text))
# product: 학습셋 제품명 중 텍스트에 등장하는 가장 긴 이름
prods = sorted(train["product"].astype(str).unique(), key=len, reverse=True)
fallback = train["product"].mode()[0]
def match_product(t):
    for p in prods:
        if p and p in str(t): return p
    return fallback
preds["product"] = [match_product(t) for t in test.text]

rows = []
for i, r in test.reset_index(drop=True).iterrows():
    for f in FIELDS:
        rows.append({"row_id": f"{r.id}__{f}", "value": preds[f][i]})
pd.DataFrame(rows).to_csv("submission.csv", index=False)
print("saved submission.csv", len(rows))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)